# Lyric Engine — Kaggle Training Notebook

**Repo:** https://github.com/SMXFREEZE/lyric-engine  
**Runtime:** GPU P100 or T4 (enable in Settings → Accelerator)

### Stages
1. Check GPU
2. Install dependencies
3. Clone / update repo from GitHub
4. Config (tokens, model, genre)
5. Scrape lyrics from Genius API
6. Build style vectors
7. Verify pipeline
8. Load model + LoRA
9. Train
10. Test generation
11. Save checkpoint

In [ ]:
# -- 1. Check GPU --
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {name}')
    print(f'VRAM : {vram:.1f} GB')
else:
    print('NO GPU — go to Settings → Accelerator → GPU T4 or P100')
    raise SystemExit('Enable GPU first')

!nvidia-smi

In [ ]:
# -- 2. Install dependencies --
!pip install -q \
    transformers peft accelerate bitsandbytes trl \
    datasets lyricsgenius \
    pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer python-dotenv
print('All dependencies installed.')

In [ ]:
# -- 3. Clone / update repo --
import os, sys

REPO = 'https://github.com/SMXFREEZE/lyric-engine'
DEST = '/kaggle/working/lyric-engine'

if os.path.exists(f'{DEST}/.git'):
    print('Already cloned — pulling latest...')
    !git -C {DEST} pull origin main
else:
    print('Cloning repo...')
    !git clone {REPO} {DEST}

%cd {DEST}
print('\nLatest commits:')
!git log --oneline -5

sys.path.insert(0, '.')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
print(f'\nCheckpoints → {CHECKPOINT_DIR}')

In [ ]:
# -- 4. Config --
# !! Add your tokens as Kaggle Secrets (eye icon on left sidebar) !!
# Secret name: GENIUS_TOKEN  → your Genius client access token
# Secret name: HF_TOKEN      → your HuggingFace token (write access)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GENIUS_TOKEN = secrets.get_secret('GENIUS_TOKEN')   # genius.com/api-clients
HF_TOKEN     = secrets.get_secret('HF_TOKEN')       # huggingface.co/settings/tokens

os.environ['GENIUS_TOKEN']            = GENIUS_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# ── Model ──────────────────────────────────────────────────────────────────────
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'   # fits in 16GB VRAM with 4-bit

# ── Training config ────────────────────────────────────────────────────────────
GENRE      = 'trap'      # trap / rnb / indie / pop / drill / alt_emo / hip_hop / country / rock / latin
BATCH_SIZE = 2
GRAD_ACCUM = 16          # effective batch = 32
EPOCHS     = 1
LR         = 2e-4
LORA_RANK  = 16
use_4bit   = True

# ── Scraper config ─────────────────────────────────────────────────────────────
MAX_SONGS_PER_ARTIST = 30   # reduce if hitting Genius rate limits

print(f'Base model : {BASE_MODEL}')
print(f'Genre      : {GENRE}')
print(f'Eff. batch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'Genius token loaded : {"yes" if GENIUS_TOKEN else "NO — add it to Kaggle Secrets"}')
print(f'HF token loaded     : {"yes" if HF_TOKEN else "NO — add it to Kaggle Secrets"}')

In [ ]:
# -- 5. Scrape lyrics from Genius --
import json, time, re
import lyricsgenius
from tqdm.auto import tqdm

GENRE_SEEDS = {
    'trap':    ['Future', 'Young Thug', 'Gunna', 'Lil Baby', '21 Savage', 'Roddy Ricch', 'Offset', 'Playboi Carti'],
    'rnb':     ['Frank Ocean', 'SZA', 'H.E.R.', 'Bryson Tiller', 'Summer Walker', 'The Weeknd', 'Brent Faiyaz'],
    'indie':   ['Phoebe Bridgers', 'Sufjan Stevens', 'Bon Iver', 'Mitski', 'Clairo', 'Soccer Mommy'],
    'pop':     ['Taylor Swift', 'Olivia Rodrigo', 'Dua Lipa', 'Ariana Grande', 'Sabrina Carpenter', 'Charli XCX'],
    'drill':   ['Pop Smoke', 'Fivio Foreign', 'Lil Durk', 'King Von', 'Central Cee', 'Digga D'],
    'alt_emo': ['Paramore', 'My Chemical Romance', 'Lana Del Rey', 'Billie Eilish', 'Conan Gray'],
    'hip_hop': ['Kendrick Lamar', 'J. Cole', 'Drake', 'Jay-Z', 'Nas', 'Lupe Fiasco'],
    'country': ['Morgan Wallen', 'Luke Combs', 'Zach Bryan', 'Kacey Musgraves', 'Tyler Childers'],
    'rock':    ['Arctic Monkeys', 'The Strokes', 'Tame Impala', 'Radiohead', 'Foo Fighters'],
    'latin':   ['Bad Bunny', 'J Balvin', 'Karol G', 'Peso Pluma', 'Rauw Alejandro'],
}

def clean_lyrics(raw):
    text = re.sub(r'\[.*?\]', '', raw)
    text = re.sub(r'\d+ Contributors?.*?Lyrics', '', text, flags=re.DOTALL)
    text = re.sub(r'\d*Embed$', '', text.strip())
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    verbose=False,
    remove_section_headers=True,
    skip_non_songs=True,
    timeout=10,
)

artists = GENRE_SEEDS.get(GENRE, GENRE_SEEDS['trap'])
out_path = f'data/raw/{GENRE}.jsonl'
songs = []

print(f'Scraping {len(artists)} artists for genre: {GENRE}')
for artist_name in tqdm(artists):
    try:
        artist = genius.search_artist(artist_name, max_songs=MAX_SONGS_PER_ARTIST, sort='popularity')
        if not artist:
            continue
        for song in artist.songs:
            if not song.lyrics or len(song.lyrics.split()) < 80:
                continue
            lyrics = clean_lyrics(song.lyrics)
            if len(lyrics.split()) < 80:
                continue
            songs.append({
                'artist': artist_name,
                'title':  song.title,
                'genre':  GENRE,
                'lyrics': lyrics,
            })
        time.sleep(1)
    except Exception as e:
        print(f'  Error scraping {artist_name}: {e}')

with open(out_path, 'w', encoding='utf-8') as f:
    for s in songs:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'\nSaved {len(songs)} songs → {out_path}')

# Fallback: if Genius failed, load from HuggingFace
if len(songs) < 20:
    print('Too few songs from Genius — loading fallback HuggingFace dataset...')
    from datasets import load_dataset
    ds = load_dataset('Cropinky/rap_lyrics_english', split='train')
    all_lines = [row['text'] for row in ds if row['text'].strip()]
    songs = []
    for i in range(0, min(len(all_lines), 40 * 2000), 40):
        chunk = all_lines[i:i+40]
        if len(chunk) < 8: continue
        songs.append({'artist': f'artist_{i//40//5:04d}', 'title': f'song_{i//40:04d}', 'genre': GENRE, 'lyrics': '\n'.join(chunk)})
    with open(out_path, 'w') as f:
        for s in songs:
            f.write(json.dumps(s) + '\n')
    print(f'Fallback: saved {len(songs)} songs')

In [ ]:
# -- 6. Build style vectors --
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=2,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# -- 7. Verify pipeline --
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
print(f"Artist : {sample['artist']}")
print(f"Title  : {sample['title']}")
print()

scheme = detect_scheme(lines)
print(f"Rhyme scheme  : {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density : {scheme['rhyme_density']}")
print()

for em in score_lyrics('\n'.join(lines)):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables:2d}syl] {em.text[:60]}")

print('\nPipeline OK ✓')

In [ ]:
# -- 8. Load base model + LoRA --
from src.model.lyrics_model import load_base_model, apply_lora, LyricsModel

device = 'cuda'

print(f'Loading {BASE_MODEL} in {"4-bit" if use_4bit else "full"} precision...')
base, tokenizer = load_base_model(BASE_MODEL, use_4bit=use_4bit, device=device)
base = apply_lora(base, rank=LORA_RANK, alpha=LORA_RANK * 2)
model = LyricsModel(base, d_model=base.config.hidden_size)

base.print_trainable_parameters()
print(f'Model loaded on: {next(base.parameters()).device}')

In [ ]:
# -- 9. Train genre LoRA --
from src.training.sft import train_sft

train_sft(
    stage=2,
    genre=GENRE,
    data_path=f'data/raw/{GENRE}.jsonl',
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=EPOCHS,
    lr=LR,
    lora_rank=LORA_RANK,
    use_4bit=use_4bit,
    style_vec_dir='data/style_vectors',
)
print('Training complete ✓')

In [ ]:
# -- 10. Test generation --
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'

print(f'Loading checkpoint from {ckpt_path}...')
tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, load_in_4bit=True, device_map='auto'
)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f'\n=== Generated {GENRE.upper()} verse ===')
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for i, line in enumerate(verse, 1):
    print(f'  {i}. {line}')

In [ ]:
# -- 11. Checkpoint summary --
import os

ckpt = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'
print(f'Checkpoint: {ckpt}')
print()
if os.path.exists(ckpt):
    total_mb = 0
    for fname in sorted(os.listdir(ckpt)):
        size = os.path.getsize(f'{ckpt}/{fname}')
        total_mb += size / 1e6
        print(f'  {fname:45s} {size/1e6:6.1f} MB')
    print(f'\n  Total: {total_mb:.0f} MB')
else:
    print('Checkpoint directory not found — did training complete?')

print()
print('To download: Kaggle → Output tab (right sidebar) → Download')